TurkQuish feature-group ablation — notebook-aligned version.

This version is based on the exact feature groups defined in the notebook:

- 58 lexical / structural features
- 38 adversarial / brand features
- 20 Turkish linguistic features
- 5 Turkish extension features
- 18 graph-based infrastructure features

Total extracted features: 139
Artifact-controlled retained features: 135
Removed artifact/leakage columns:
    is_tr_domain
    is_https
    campaign_token_score
    cluster_malicious_ratio

Recommended input:
    features_full_final_inductive_dedup.csv

This deduplicated file should keep the inductive graph values and remove duplicate
columns such as .1 and .2 suffixes.


In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    matthews_corrcoef,
    accuracy_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATH CONFIGURATION
# ============================================================

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"

INPUT_CANDIDATES = [
    "features_full_final_inductive_dedup.csv",
    "features_full_final_inductive.csv",
    "features_full_final.csv",
    "features_full_v12.csv",
]

OUTPUT_DIR = os.path.join(DATA_DIR, "ablation_outputs")
PLOT_DIR = os.path.join(DATA_DIR, "ablation_plots")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

SEED = 42


# ============================================================
# 2. EXACT FEATURE LISTS FROM THE NOTEBOOK
# ============================================================

LEXICAL_58 = [
    "url_len", "num_dots", "num_slashes", "num_digits", "num_specials",
    "subdomain_len", "domain_len", "tld_len", "is_tr_domain",
    "digit_ratio", "alpha_ratio", "special_ratio",
    "num_subdomains", "has_www", "subdomain_digits",
    "path_len", "path_depth", "path_digits", "has_php", "has_exe",
    "has_query", "query_len", "num_params",
    "num_hyphens", "num_underscores", "num_at_signs", "num_ampersands",
    "num_equals", "num_percent", "has_at_in_url", "has_port", "has_ip",
    "is_https", "has_double_slash",
    "url_entropy", "domain_entropy", "path_entropy",
    "has_repeated_chars", "has_hex_encoding", "num_encoded_chars", "has_double_dot",
    "has_turkish_keyword", "num_turkish_keywords",
    "has_phishing_keyword", "num_phishing_keywords",
    "has_malware_keyword", "num_malware_keywords",
    "has_scam_keyword", "num_scam_keywords",
    "domain_hyphen_count", "domain_digit_count", "domain_has_number",
    "is_suspicious_tld", "is_free_hosting", "url_len_bucket",
    "num_tokens", "max_token_len", "mean_token_len",
]

ADVERSARIAL_38 = [
    "contains_brand", "brand_in_subdomain", "brand_in_path", "brand_not_in_domain",
    "brand_tld_mismatch", "num_brands_mentioned", "brand_with_hyphen",
    "brand_plus_keyword", "min_brand_edit_dist", "is_typo_squat",
    "has_char_substitution", "has_doubled_char", "brand_homoglyph",
    "excessive_subdomains", "tr_in_subdomain", "com_in_subdomain",
    "brand_dot_in_subdomain", "deep_subdomain_nesting",
    "susp_tld_with_brand", "susp_tld_with_keyword",
    "short_domain_susp_tld", "numeric_with_susp_tld",
    "pct_encoded_ratio", "has_unicode_escape", "has_punycode",
    "hex_in_domain", "excessive_hyphens", "random_looking_domain",
    "consonant_heavy_domain", "long_random_path", "hash_like_segment",
    "base64_like_segment", "many_path_dirs", "suspicious_file_in_path",
    "has_url_in_url", "has_redirect_param", "at_symbol_trick", "double_protocol",
]

TR_LING_20 = [
    "tr_char_count", "tr_char_ratio", "has_tr_char", "tr_char_in_domain",
    "tr_char_in_path", "distinct_tr_chars", "tr_stopword_count",
    "tr_stopword_ratio", "tr_suffix_count", "vowel_harmony_score",
    "tr_token_ratio", "tr_bigram_score", "tr_vs_en_bigram",
    "langid_tr_confidence", "is_turkish_dominant",
    "has_tr_bank_term", "has_tr_gov_term", "has_tr_ecommerce_term",
    "has_tr_telecom_term", "tr_sector_count",
]

TR_EXT_5 = [
    "tr_phishing_vocab_score", "tr_transliteration_score",
    "tr_brand_impersonation_score", "tr_semantic_urgency_score",
    "tr_scam_vocab_score",
]

GRAPH_18 = [
    "rare_token_count", "max_token_cluster_size", "shared_token_degree",
    "campaign_token_score", "unique_token_ratio", "token_reuse_score",
    "domain_family_size", "tld_token_cooccur", "sibling_domain_count",
    "domain_ngram_cluster", "registrant_pattern_score",
    "subdomain_reuse_count", "path_template_reuse", "host_pattern_degree",
    "campaign_membership", "token_centrality", "is_hub_domain",
    "cluster_malicious_ratio",
]

DOCUMENTED_139 = LEXICAL_58 + ADVERSARIAL_38 + TR_LING_20 + TR_EXT_5 + GRAPH_18

assert len(DOCUMENTED_139) == 139, f"Expected 139 features, got {len(DOCUMENTED_139)}"


# ============================================================
# 3. METADATA AND ARTIFACT COLUMNS
# ============================================================

META_COLS = {
    "url", "source", "tld", "label", "label_enc", "class_final",
    "class_enc", "fold", "reg_domain", "registered_domain",
}

ARTIFACT_COLUMNS = {
    "is_tr_domain",
    "is_https",
    "campaign_token_score",
    "cluster_malicious_ratio",
}

CLASS_NAME_ORDER = ["benign", "phishing", "malware", "scam", "other_malicious"]


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def resolve_input_file(data_dir, candidates):
    for name in candidates:
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            print(f"[INFO] Using input file: {path}")
            return path

    raise FileNotFoundError(
        "No input feature file found. Expected one of:\n" +
        "\n".join(os.path.join(data_dir, c) for c in candidates)
    )


def load_dataset():
    input_path = resolve_input_file(DATA_DIR, INPUT_CANDIDATES)
    df = pd.read_csv(input_path, low_memory=False)
    print(f"[INFO] Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df, input_path


def verify_columns(df):
    required = {"class_enc", "fold"}
    missing_required = required - set(df.columns)

    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    present_features = [f for f in DOCUMENTED_139 if f in df.columns]
    missing_features = [f for f in DOCUMENTED_139 if f not in df.columns]

    print("\n[FEATURE AUDIT]")
    print(f"Documented features expected: {len(DOCUMENTED_139)}")
    print(f"Documented features present:  {len(present_features)}")

    if missing_features:
        print("\n[WARNING] Missing documented features:")
        for f in missing_features:
            print(f"  - {f}")

    suffixed = [c for c in df.columns if c.endswith(".1") or c.endswith(".2")]
    if suffixed:
        print("\n[WARNING] Suffixed duplicate columns found.")
        print("You should run the notebook dedup step and use features_full_final_inductive_dedup.csv")
        print(f"Number of suffixed columns: {len(suffixed)}")

    return present_features


def remove_artifacts(features):
    return [f for f in features if f not in ARTIFACT_COLUMNS]


def available(features, df):
    return [f for f in features if f in df.columns]


def binary_projection(y):
    return (np.asarray(y) != 0).astype(int)


def safe_auc(y_true_binary, y_score_malicious):
    try:
        return roc_auc_score(y_true_binary, y_score_malicious)
    except Exception:
        return np.nan


def make_model():
    """
    Notebook-aligned HistGB reference configuration.

    This follows the notebook's deployment configuration:
        max_depth=6
        learning_rate=0.05
        max_iter=300
    """
    clf = HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.05,
        max_iter=300,
        random_state=SEED,
    )

    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", clf),
        ]
    )


def evaluate_fold(X_train, y_train, X_test, y_test):
    model = make_model()

    fit_start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - fit_start

    pred_start = time.perf_counter()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    pred_time = time.perf_counter() - pred_start

    y_test_bin = binary_projection(y_test)
    y_pred_bin = binary_projection(y_pred)

    # Probability of malicious = 1 - probability of benign class.
    classes = list(model.named_steps["classifier"].classes_)
    benign_index = classes.index(0) if 0 in classes else None

    if benign_index is not None:
        y_score_malicious = 1.0 - y_proba[:, benign_index]
    else:
        y_score_malicious = y_pred_bin

    metrics = {
        "binary_auc": safe_auc(y_test_bin, y_score_malicious),
        "binary_f1": f1_score(y_test_bin, y_pred_bin, zero_division=0),
        "binary_mcc": matthews_corrcoef(y_test_bin, y_pred_bin),
        "five_class_accuracy": accuracy_score(y_test, y_pred),
        "five_class_macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "five_class_weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "fit_time_sec": fit_time,
        "predict_time_sec": pred_time,
        "inference_ms_per_url": (pred_time / len(X_test)) * 1000,
    }

    return metrics


def summarize(rows):
    df = pd.DataFrame(rows)

    metric_cols = [
        "binary_auc",
        "binary_f1",
        "binary_mcc",
        "five_class_accuracy",
        "five_class_macro_f1",
        "five_class_weighted_f1",
        "fit_time_sec",
        "predict_time_sec",
        "inference_ms_per_url",
    ]

    out = {}
    for col in metric_cols:
        out[f"{col}_mean"] = df[col].mean()
        out[f"{col}_std"] = df[col].std(ddof=1)

    return out


def mean_std(mean, std, decimals=4):
    if pd.isna(mean):
        return "NA"
    if pd.isna(std):
        std = 0.0
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"


def save_plot(name):
    png = os.path.join(PLOT_DIR, f"{name}.png")
    pdf = os.path.join(PLOT_DIR, f"{name}.pdf")

    plt.tight_layout()
    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    plt.close()

    print(f"[SAVED] {png}")
    print(f"[SAVED] {pdf}")


# ============================================================
# 5. BUILD ABLATION SETTINGS
# ============================================================

def build_ablation_settings(df):
    lexical = remove_artifacts(available(LEXICAL_58, df))
    adversarial = remove_artifacts(available(ADVERSARIAL_38, df))
    turkish = remove_artifacts(available(TR_LING_20 + TR_EXT_5, df))
    graph = remove_artifacts(available(GRAPH_18, df))

    full_retained = remove_artifacts(available(DOCUMENTED_139, df))

    print("\n[GROUP COUNTS AFTER ARTIFACT REMOVAL]")
    print(f"Lexical / structural:     {len(lexical)}")
    print(f"Adversarial / brand:      {len(adversarial)}")
    print(f"Turkish linguistic + ext: {len(turkish)}")
    print(f"Graph infrastructure:     {len(graph)}")
    print(f"Full retained:            {len(full_retained)}")

    # Expected after artifact control:
    # 139 extracted - 4 artifacts = 135 retained.
    if len(full_retained) != 135:
        print("\n[WARNING] Full retained feature count is not 135.")
        print("Check whether the deduplicated inductive file is being used.")

    settings = [
        {
            "setting": "A1",
            "description": "Lexical + structural only",
            "lexical_structural": True,
            "brand_adversarial": False,
            "turkish_linguistic": False,
            "graph_infrastructure": False,
            "features": lexical,
        },
        {
            "setting": "A2",
            "description": "Lexical + structural + adversarial/brand",
            "lexical_structural": True,
            "brand_adversarial": True,
            "turkish_linguistic": False,
            "graph_infrastructure": False,
            "features": sorted(set(lexical + adversarial)),
        },
        {
            "setting": "A3",
            "description": "Lexical + structural + Turkish linguistic",
            "lexical_structural": True,
            "brand_adversarial": False,
            "turkish_linguistic": True,
            "graph_infrastructure": False,
            "features": sorted(set(lexical + turkish)),
        },
        {
            "setting": "A4",
            "description": "Lexical + structural + graph-based infrastructure",
            "lexical_structural": True,
            "brand_adversarial": False,
            "turkish_linguistic": False,
            "graph_infrastructure": True,
            "features": sorted(set(lexical + graph)),
        },
        {
            "setting": "A5",
            "description": "Lexical + structural + adversarial/brand + Turkish linguistic",
            "lexical_structural": True,
            "brand_adversarial": True,
            "turkish_linguistic": True,
            "graph_infrastructure": False,
            "features": sorted(set(lexical + adversarial + turkish)),
        },
        {
            "setting": "A6",
            "description": "Lexical + structural + adversarial/brand + Turkish linguistic + graph",
            "lexical_structural": True,
            "brand_adversarial": True,
            "turkish_linguistic": True,
            "graph_infrastructure": True,
            "features": sorted(set(lexical + adversarial + turkish + graph)),
        },
        {
            "setting": "A7",
            "description": "Full retained feature set, artifact-controlled",
            "lexical_structural": True,
            "brand_adversarial": True,
            "turkish_linguistic": True,
            "graph_infrastructure": True,
            "features": full_retained,
        },
        {
            "setting": "A8",
            "description": "Full retained feature set, artifact-controlled, inductive graph projection",
            "lexical_structural": True,
            "brand_adversarial": True,
            "turkish_linguistic": True,
            "graph_infrastructure": True,
            "features": full_retained,
        },
    ]

    return settings


# ============================================================
# 6. RUN ABLATION
# ============================================================

def run_ablation(df):
    verify_columns(df)

    settings = build_ablation_settings(df)

    y = df["class_enc"].values
    folds = df["fold"].values
    fold_ids = sorted(pd.unique(folds))

    all_fold_rows = []
    summary_rows = []

    for setting in settings:
        feature_cols = setting["features"]

        if len(feature_cols) == 0:
            print(f"[SKIP] {setting['setting']} has no selected features.")
            continue

        print("\n" + "=" * 80)
        print(f"Running {setting['setting']} — {setting['description']}")
        print(f"Features: {len(feature_cols)}")
        print("=" * 80)

        setting_rows = []

        for fid in fold_ids:
            train_idx = folds != fid
            test_idx = folds == fid

            X_train = df.loc[train_idx, feature_cols].values
            y_train = y[train_idx]

            X_test = df.loc[test_idx, feature_cols].values
            y_test = y[test_idx]

            metrics = evaluate_fold(X_train, y_train, X_test, y_test)

            row = {
                "setting": setting["setting"],
                "description": setting["description"],
                "fold": fid,
                "num_features": len(feature_cols),
                "lexical_structural": setting["lexical_structural"],
                "brand_adversarial": setting["brand_adversarial"],
                "turkish_linguistic": setting["turkish_linguistic"],
                "graph_infrastructure": setting["graph_infrastructure"],
                **metrics,
            }

            setting_rows.append(row)
            all_fold_rows.append(row)

            print(
                f"Fold {fid}: "
                f"AUC={metrics['binary_auc']:.4f}, "
                f"MCC={metrics['binary_mcc']:.4f}, "
                f"Macro-F1={metrics['five_class_macro_f1']:.4f}, "
                f"Inference={metrics['inference_ms_per_url']:.4f} ms/url"
            )

        summary = summarize(setting_rows)

        summary_row = {
            "setting": setting["setting"],
            "description": setting["description"],
            "num_features": len(feature_cols),
            "lexical_structural": setting["lexical_structural"],
            "brand_adversarial": setting["brand_adversarial"],
            "turkish_linguistic": setting["turkish_linguistic"],
            "graph_infrastructure": setting["graph_infrastructure"],
            **summary,
        }

        summary_rows.append(summary_row)

    fold_df = pd.DataFrame(all_fold_rows)
    summary_df = pd.DataFrame(summary_rows)

    baseline = summary_df.loc[
        summary_df["setting"] == "A1",
        "five_class_macro_f1_mean",
    ]

    if len(baseline) > 0:
        baseline_value = baseline.iloc[0]
        summary_df["delta_macro_f1_vs_A1"] = (
            summary_df["five_class_macro_f1_mean"] - baseline_value
        )
    else:
        summary_df["delta_macro_f1_vs_A1"] = np.nan

    fold_path = os.path.join(OUTPUT_DIR, "ablation_fold_results.csv")
    summary_path = os.path.join(OUTPUT_DIR, "ablation_summary.csv")
    table_path = os.path.join(OUTPUT_DIR, "ablation_manuscript_table.csv")
    latex_path = os.path.join(OUTPUT_DIR, "ablation_table_latex.txt")

    fold_df.to_csv(fold_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    manuscript_table = pd.DataFrame({
        "Feature setting": summary_df["setting"] + ": " + summary_df["description"],
        "Lexical/structural": summary_df["lexical_structural"].map({True: "✓", False: "—"}),
        "Brand/adversarial": summary_df["brand_adversarial"].map({True: "✓", False: "—"}),
        "Turkish linguistic": summary_df["turkish_linguistic"].map({True: "✓", False: "—"}),
        "Graph infrastructure": summary_df["graph_infrastructure"].map({True: "✓", False: "—"}),
        "No. features": summary_df["num_features"],
        "Binary AUC": [
            mean_std(m, s)
            for m, s in zip(summary_df["binary_auc_mean"], summary_df["binary_auc_std"])
        ],
        "Binary F1": [
            mean_std(m, s)
            for m, s in zip(summary_df["binary_f1_mean"], summary_df["binary_f1_std"])
        ],
        "Binary MCC": [
            mean_std(m, s)
            for m, s in zip(summary_df["binary_mcc_mean"], summary_df["binary_mcc_std"])
        ],
        "Five-class macro-F1": [
            mean_std(m, s)
            for m, s in zip(summary_df["five_class_macro_f1_mean"], summary_df["five_class_macro_f1_std"])
        ],
        "Five-class weighted-F1": [
            mean_std(m, s)
            for m, s in zip(summary_df["five_class_weighted_f1_mean"], summary_df["five_class_weighted_f1_std"])
        ],
        "Δ macro-F1 vs A1": [
            f"{v:+.4f}" if not pd.isna(v) else "NA"
            for v in summary_df["delta_macro_f1_vs_A1"]
        ],
        "Inference ms/URL": [
            mean_std(m, s, decimals=4)
            for m, s in zip(summary_df["inference_ms_per_url_mean"], summary_df["inference_ms_per_url_std"])
        ],
    })

    manuscript_table.to_csv(table_path, index=False)

    latex = manuscript_table.to_latex(
        index=False,
        escape=False,
        column_format="p{4.5cm}cccccccc",
    )

    with open(latex_path, "w", encoding="utf-8") as f:
        f.write("% Feature-group ablation table for TurkQuish\n")
        f.write("% Results are mean ± std across five domain-insulated folds.\n")
        f.write(latex)

    print("\n[SAVED OUTPUTS]")
    print(f"- {fold_path}")
    print(f"- {summary_path}")
    print(f"- {table_path}")
    print(f"- {latex_path}")

    return fold_df, summary_df, manuscript_table


# ============================================================
# 7. PLOTS
# ============================================================

def short_label(row):
    mapping = {
        "A1": "A1\nLexical",
        "A2": "A2\n+ Brand",
        "A3": "A3\n+ Turkish",
        "A4": "A4\n+ Graph",
        "A5": "A5\n+ Brand\n+ Turkish",
        "A6": "A6\n+ Brand\n+ Turkish\n+ Graph",
        "A7": "A7\nFull retained",
        "A8": "A8\nFull retained\ninductive",
    }
    return mapping.get(row["setting"], row["setting"])


def plot_results(summary_df):
    plot_df = summary_df.copy()
    plot_df["short_label"] = plot_df.apply(short_label, axis=1)

    # Plot 1 — Macro-F1
    plt.figure(figsize=(11, 5.5))
    x = np.arange(len(plot_df))
    y = plot_df["five_class_macro_f1_mean"]

    plt.bar(x, y)
    plt.xticks(x, plot_df["short_label"], rotation=0)
    plt.ylabel("Five-class macro-F1")
    plt.title("Feature-group ablation: five-class macro-F1")
    plt.grid(axis="y", alpha=0.25)

    for i, v in enumerate(y):
        plt.text(i, v + 0.002, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

    save_plot("fig_ablation_macro_f1")

    # Plot 2 — Delta macro-F1
    plt.figure(figsize=(11, 5.5))
    delta = plot_df["delta_macro_f1_vs_A1"]

    plt.axhline(0, linewidth=1)
    plt.bar(x, delta)
    plt.xticks(x, plot_df["short_label"], rotation=0)
    plt.ylabel("Δ macro-F1 vs lexical baseline")
    plt.title("Incremental contribution of URL-only feature groups")
    plt.grid(axis="y", alpha=0.25)

    for i, v in enumerate(delta):
        plt.text(
            i,
            v + (0.001 if v >= 0 else -0.001),
            f"{v:+.4f}",
            ha="center",
            va="bottom" if v >= 0 else "top",
            fontsize=9,
        )

    save_plot("fig_ablation_delta_macro_f1")

    # Plot 3 — Multi-metric comparison
    metrics = [
        ("binary_auc_mean", "Binary AUC"),
        ("binary_mcc_mean", "Binary MCC"),
        ("five_class_macro_f1_mean", "Macro-F1"),
        ("five_class_weighted_f1_mean", "Weighted-F1"),
    ]

    plt.figure(figsize=(12, 6))
    width = 0.18

    for j, (col, label) in enumerate(metrics):
        offset = (j - 1.5) * width
        plt.bar(x + offset, plot_df[col], width=width, label=label)

    plt.xticks(x, plot_df["short_label"], rotation=0)
    plt.ylabel("Score")
    plt.title("Ablation comparison across evaluation metrics")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", alpha=0.25)
    plt.legend(ncol=2)

    save_plot("fig_ablation_metrics_comparison")

    # Plot 4 — Efficiency trade-off
    plt.figure(figsize=(8.5, 5.5))

    plt.scatter(
        plot_df["inference_ms_per_url_mean"],
        plot_df["five_class_macro_f1_mean"],
        s=90,
    )

    for _, row in plot_df.iterrows():
        plt.annotate(
            row["setting"],
            (row["inference_ms_per_url_mean"], row["five_class_macro_f1_mean"]),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=9,
        )

    plt.xlabel("Inference time per URL (ms)")
    plt.ylabel("Five-class macro-F1")
    plt.title("Accuracy–efficiency trade-off across feature settings")
    plt.grid(alpha=0.25)

    save_plot("fig_ablation_efficiency_tradeoff")

    # Plot 5 — Feature group inclusion matrix
    group_cols = [
        "lexical_structural",
        "brand_adversarial",
        "turkish_linguistic",
        "graph_infrastructure",
    ]

    group_labels = [
        "Lexical /\nstructural",
        "Brand /\nadversarial",
        "Turkish\nlinguistic",
        "Graph\ninfrastructure",
    ]

    matrix = plot_df[group_cols].astype(int).values

    plt.figure(figsize=(8.5, 5.8))
    plt.imshow(matrix, aspect="auto", interpolation="nearest")

    plt.yticks(np.arange(len(plot_df)), plot_df["short_label"])
    plt.xticks(np.arange(len(group_labels)), group_labels)
    plt.title("Feature groups included in each ablation setting")

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            text = "✓" if matrix[i, j] == 1 else "—"
            plt.text(j, i, text, ha="center", va="center", fontsize=14)

    plt.xlabel("Feature group")
    plt.ylabel("Ablation setting")

    save_plot("fig_ablation_feature_group_matrix")


# ============================================================
# 8. MAIN
# ============================================================

if __name__ == "__main__":
    df, input_path = load_dataset()

    fold_df, summary_df, manuscript_table = run_ablation(df)

    plot_results(summary_df)

    print("\nDONE ✅")
    print("Use these manuscript files:")
    print(f"  {os.path.join(OUTPUT_DIR, 'ablation_manuscript_table.csv')}")
    print(f"  {os.path.join(OUTPUT_DIR, 'ablation_table_latex.txt')}")
    print("\nUse these figures:")
    print(f"  {os.path.join(PLOT_DIR, 'fig_ablation_delta_macro_f1.pdf')}")
    print(f"  {os.path.join(PLOT_DIR, 'fig_ablation_macro_f1.pdf')}")
    print(f"  {os.path.join(PLOT_DIR, 'fig_ablation_metrics_comparison.pdf')}")
    print(f"  {os.path.join(PLOT_DIR, 'fig_ablation_efficiency_tradeoff.pdf')}")
    print(f"  {os.path.join(PLOT_DIR, 'fig_ablation_feature_group_matrix.pdf')}")

[INFO] Using input file: /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/features_full_final_inductive_dedup.csv
[INFO] Loaded dataset: 1,239,308 rows × 148 columns

[FEATURE AUDIT]
Documented features expected: 139
Documented features present:  139

[GROUP COUNTS AFTER ARTIFACT REMOVAL]
Lexical / structural:     56
Adversarial / brand:      38
Turkish linguistic + ext: 25
Graph infrastructure:     16
Full retained:            135

Running A1 — Lexical + structural only
Features: 56
Fold 0: AUC=0.9953, MCC=0.9088, Macro-F1=0.8564, Inference=0.0860 ms/url
Fold 1: AUC=0.9948, MCC=0.8680, Macro-F1=0.8410, Inference=0.0859 ms/url
Fold 2: AUC=0.9971, MCC=0.9237, Macro-F1=0.8614, Inference=0.0887 ms/url
Fold 3: AUC=0.9976, MCC=0.9331, Macro-F1=0.8572, Inference=0.0843 ms/url
Fold 4: AUC=0.9934, MCC=0.8709, Macro-F1=0.8404, Inference=0.0887 ms/url

Running A2 — Lexical + structural + adversarial/brand
Features: 94
Fold 0: AUC=0.9958, MCC=0.9180, Macro-F1=0.8775, Inference=0.08